
## Introduction
---
As a machine learning engineer working for a social media company, your responsibility is to assess whether songs shared on the platform are suitable for children. Since evaluating every song using large language models (LLMs) is computationally expensive, an alternative approach using Retrieval-Augmented Generation (RAG) is introduced.

RAG works by combining a retrieval component, which finds relevant information (such as embeddings of previously answered questions about content safety), with a generation component that uses this retrieved context to make predictions about new songs. This method allows the system to scale efficiently while still ensuring that each song is properly evaluated for child safety, without needing to run a full LLM inference for every single case.

---

### Import Libraries

In [13]:
import torch
import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from transformers import BertTokenizer, BertModel


### Helper Function

In [2]:
def tsne_plot(data, plot):
    
    tsne = TSNE(n_components=3, random_state=42, perplexity=(50,data.shape[0]-1))
    data_3d = tsne.fit_transform(data)
    
    fig = plt.figure(figsize=(8,8))
    ax = fig.add_subplot(111, projection='3d')
    
    color = plt.cm.rainbow(np.linspace(0,1,len(data_3d)))
    
    for idx, point, in zip(range(len(data_3d)), data_3d):
        ax.scatter(point[0], point[1], point[2], color= color[idx], label = f'{plot} {idx+1}')
    
    ax.set_xlabel("TSNE component 1")
    ax.set_ylabel("TSNE Component 2")
    ax.set_zlabel("TSNE Component 3")
    
    plt.title("3D-TSNE Visualization of "+ plot+ " Embeddings")
    plt.legend(title = plot + "index", bbox_to_anchor = (1.05,1), loc = 'upper left')
    plt.show()

### Tokenizer and Model

In [3]:
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)

We consider the input as a list of tuple

In [4]:
# Input text to get embeddings for
input_text = [("This is an example sentence for BERT embeddings.", "How do you like it "),("There are other models")]

We use **batch_encode_plus** method for tokenizing the text. It automatically handles the padding and truncation to ensure uniformity in input length, which is difficult for batch processing like BERT models.

In [5]:
tokenized_input = tokenizer(input_text, padding=True, truncation=True, add_special_tokens=True)
tokenized_input

{'input_ids': [[101, 2023, 2003, 2019, 2742, 6251, 2005, 14324, 7861, 8270, 4667, 2015, 1012, 102, 2129, 2079, 2017, 2066, 2009, 102], [101, 2045, 2024, 2060, 4275, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}

In [7]:
tokenized_input['input_ids'][0]

[101,
 2023,
 2003,
 2019,
 2742,
 6251,
 2005,
 14324,
 7861,
 8270,
 4667,
 2015,
 1012,
 102,
 2129,
 2079,
 2017,
 2066,
 2009,
 102]

In [8]:
# confirm tokenization by verification
decode_input = tokenizer.decode(tokenized_input['input_ids'][0])
decode_input

'[CLS] this is an example sentence for bert embeddings. [SEP] how do you like it [SEP]'

In [9]:
print(f"length {len(decode_input.split())}")

length 16


In [10]:
## Define Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device.type

'cuda'

In [12]:
input_ids_tensor = torch.tensor(tokenized_input['input_ids']).to(device)
attention_tensor = torch.tensor(tokenized_input['attention_mask']).to(device)
input_ids_tensor, attention_tensor

(tensor([[  101,  2023,  2003,  2019,  2742,  6251,  2005, 14324,  7861,  8270,
           4667,  2015,  1012,   102,  2129,  2079,  2017,  2066,  2009,   102],
         [  101,  2045,  2024,  2060,  4275,   102,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0]],
        device='cuda:0'),
 tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]],
        device='cuda:0'))

In [14]:
model = BertModel.from_pretrained(model_name)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vish\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
model.to(device)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [16]:
word_embeddings = model(input_ids_tensor, attention_tensor)
word_embeddings

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-0.1325, -0.3176, -0.0040,  ..., -0.3610,  0.1377,  0.7386],
         [-0.8879, -0.6018, -0.2917,  ..., -0.4431,  0.5632,  0.3510],
         [-0.3720, -0.8579,  0.2728,  ..., -0.1236,  0.3188,  0.8310],
         ...,
         [ 0.6068,  0.0490,  0.4654,  ..., -0.0692, -0.0382,  0.5104],
         [-0.1788, -1.1978, -0.2283,  ..., -0.1520, -0.4289,  0.0148],
         [ 0.6963,  0.1135, -0.4511,  ...,  0.4190, -0.8829, -0.2686]],

        [[-0.1595,  0.0046,  0.1481,  ..., -0.0987,  0.2391,  0.3857],
         [-0.2603, -0.0485,  0.0093,  ..., -0.1763,  0.6517,  0.0109],
         [-0.5163, -0.4118,  0.5324,  ..., -0.3391,  0.4124,  0.3600],
         ...,
         [ 0.1249,  0.0976,  0.4567,  ..., -0.0268,  0.1607,  0.1392],
         [-0.5925, -0.7013,  0.1059,  ...,  0.4546,  0.3088,  0.1715],
         [-0.3717, -0.5228,  0.2766,  ...,  0.2273,  0.1079,  0.0727]]],
       device='cuda:0', grad_fn=<NativeLayerNormBackw